# Original coursework interaction analysis
Output-cleared archive. In-sample fitted probabilities are exploratory, not validated individual risk predictions. Associations do not establish causes. Use src/analyze.py for reproducible aggregate analysis.

对于非加班员工而言，更高的工作与生活平衡度既可能意味着舒适，也可能暗示停滞——即处于稳定且低压力的岗位，却面临有限的挑战或晋升空间。随着平衡度的提升，这类员工可能开始探索外部发展机会，这解释了为何非加班群体中工作与生活平衡度提升时离职率随之上升。
然而对加班员工而言，工作与生活平衡的结构性缺陷源于其工作时间的本质。他们的基础平衡水平长期处于低位，因此感知平衡的微小变化对其去留决策影响甚微。结果是他们的离职率曲线保持平稳：加班本身——而非工作生活平衡——才是决定人员流失风险的主导因素。
For non-overtime employees, higher work–life balance can signal comfort but also stagnation—roles that are stable and low-pressure yet offer limited challenge or advancement. As balance improves, these employees may begin exploring external opportunities for growth, which explains why attrition rises with WLB in the non-overtime group.
For overtime employees, however, work–life balance is already structurally compromised by the nature of their schedule. Their baseline WLB is chronically low, so incremental changes in perceived balance have little impact on their decision to stay or leave. As a result, their attrition line remains flat: overtime itself—not WLB—is the dominant factor shaping turnover risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

df = pd.read_csv("employee_attrition_dataset.csv")
df["Attrition_Flag"] = (df["Attrition"] == "Yes").astype(int)
df["OT"] = (df["Overtime"] == "Yes").astype(int)

# Logit model with interaction
model_ot_wlb = smf.logit(
    formula="Attrition_Flag ~ OT * Work_Life_Balance",
    data=df
).fit()

print(model_ot_wlb.summary())

# Predict
df["pred_prob_ot_wlb"] = model_ot_wlb.predict(df)

# Pivot for plotting (same structure as your example)
lind_ot_wlb = (
    df.groupby(["Work_Life_Balance", "OT"])
      .agg({"pred_prob_ot_wlb": "mean"})
      .reset_index()
)

# Plot
plt.figure(figsize=(9,6))

for ot_val in [0, 1]:
    sub = lind_ot_wlb[lind_ot_wlb["OT"] == ot_val]
    plt.plot(
        sub["Work_Life_Balance"],
        sub["pred_prob_ot_wlb"],
        marker="o",
        label=f"Overtime={ot_val}"
    )

plt.xlabel("Work-Life Balance")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability by Work-Life Balance × Overtime (with interaction)")
plt.legend(title="Overtime")
plt.tight_layout()
plt.savefig("Predicted Attrition Probability by Work-Life Balance × Overtime (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()

收入增长在不同职级员工中传递着不同信号。
对于一级和二级员工，更高收入往往意味着他们已达到薪酬区间的上限；这非但无法留住他们，反而增强了他们的自信和市场竞争力，促使他们离职寻求更高职位。
对于五级员工，收入增长同样会加剧流失率，因为这些资深员工流动性极高；更高薪资增强了他们的外部议价能力，为其打开了市场中高薪管理职位的大门。
相比之下，中层员工（三级至四级）将收入增长视为晋升信号和组织投资。在此层级，更高薪资意味着稳定性、认可度和清晰的职业路径，从而降低流失率。
由此形成鲜明模式：收入增长在层级结构的底层和顶层引发人员流动，却能稳固中层员工队伍。
Income increases send different signals at different job levels.
For Level-1 and Level-2 employees, higher income often indicates they have reached the upper bound of their pay band; instead of making them stay, it boosts their confidence and marketability, encouraging them to leave for higher-level roles elsewhere.
For Level-5 employees, income growth similarly increases attrition because these senior employees are highly mobile; higher pay strengthens their external bargaining power and opens doors to lucrative leadership opportunities in the market.
In contrast, mid-level employees (Levels 3–4) interpret rising income as a sign of progression and organizational investment. Higher pay here signals stability, recognition, and a clear career path, reducing attrition.
This creates the distinct pattern: income growth drives turnover at the bottom and top of the hierarchy but retains employees in the middle.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

df = pd.read_csv("employee_attrition_dataset.csv")

df["Attrition_Flag"] = (df["Attrition"] == "Yes").astype(int)

model_inc_lvl = smf.logit(
    formula="Attrition_Flag ~ Monthly_Income * C(Job_Level)",
    data=df
).fit()

print(model_inc_lvl.summary())

df["pred_prob_inc_lvl"] = model_inc_lvl.predict(df)

df["Income_Bin"] = pd.qcut(
    df["Monthly_Income"],
    q=20, 
    duplicates="drop"
)

df["Income_Mid"] = df["Income_Bin"].apply(lambda x: x.mid)

lind_inc_lvl = (
    df.groupby(["Income_Mid", "Job_Level"])
      .agg({"pred_prob_inc_lvl": "mean"})
      .reset_index()
)

plt.figure(figsize=(9,6))

for lvl in sorted(df["Job_Level"].unique()):
    sub = lind_inc_lvl[lind_inc_lvl["Job_Level"] == lvl].sort_values("Income_Mid")
    plt.plot(
        sub["Income_Mid"],
        sub["pred_prob_inc_lvl"],
        marker="o",
        markersize=6,
        label=f"Job Level={lvl}"
    )

plt.xlabel("Monthly Income (binned midpoint)")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability by Monthly Income × Job Level (with interaction)")
plt.legend(title="Job Level")
plt.tight_layout()
plt.savefig("Predicted Attrition Probability by Monthly Income × Job Level (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()

Early-tenure employees (0–5 years) are still in their learning and proving stage — their expectations are low, turnover costs are low, and they are mainly focused on building skills rather than chasing promotions, so promotion delays don’t push them out.
Long-tenure employees (11+ years) are deeply embedded in the organization; their networks, routines, and job security make them less sensitive to promotion timing.
But mid-career employees (6–10 years) are at the point where they expect upward mobility — they have invested enough and want a return, but are still mobile enough to leave — so a long gap without promotion becomes a clear signal of stalled progression, making this group uniquely likely to quit when promotion is delayed.早期终身职位员工（0-5年）仍处于学习和证明阶段——他们的期望值较低，离职成本较低，他们主要关注的是培养技能，而不是追求升职，因此升职延迟不会把他们赶出去。
长期员工（11年以上）深深扎根于组织；他们的人际网络、日常生活和工作保障使他们对晋升时机不那么敏感。
但处于职业生涯中期的员工（6-10年）正处于他们期望向上流动的阶段——他们已经投入了足够多的时间精力，希望得到回报，但仍然有足够的流动性离开——所以很低

In [ ]:
# Tenure grouping
def tenure_group(x):
    if x <= 5:
        return "0-5"
    elif x <= 10:
        return "6-10"
    elif x <= 15:
        return "11-15"
    elif x <= 20:
        return "16-20"
    elif x <= 25:
        return "21-25"
    else:
        return "26-30"

df["Tenure_Group"] = df["Years_at_Company"].apply(tenure_group)

# Logit model with interaction
model_prom_ten = smf.logit(
    formula="Attrition_Flag ~ Years_Since_Last_Promotion * Tenure_Group",
    data=df
).fit()

print(model_prom_ten.summary())

# Predict
df["pred_prob_prom_ten"] = model_prom_ten.predict(df)

# Group for plotting
lind_prom_ten = (
    df.groupby(["Years_Since_Last_Promotion", "Tenure_Group"])
      .agg({"pred_prob_prom_ten": "mean"})
      .reset_index()
)

# Plot same style as your reference
plt.figure(figsize=(9,6))

for tg in ["0-5", "6-10", "11-15", "16-20", "21-25", "26-30"]:
    sub = lind_prom_ten[lind_prom_ten["Tenure_Group"] == tg]
    plt.plot(
        sub["Years_Since_Last_Promotion"],
        sub["pred_prob_prom_ten"],
        marker="o",
        label=f"Tenure={tg}"
    )

plt.xlabel("Years Since Last Promotion")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability by Promotion Stagnation × Tenure Group (with interaction)")
plt.legend(title="Tenure Group")
plt.tight_layout()
plt.savefig("Predicted Attrition Probability by Promotion Stagnation × Tenure Group (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt


df = pd.read_csv("employee_attrition_dataset.csv")

df["Attrition_Flag"] = (df["Attrition"] == "Yes").astype(int)

print(df.columns)

图一 高满意度 + 经理关系好的人，估计更容易被别的公司挖走。当员工与经理的关系不好时，更高的工作满意度会放大他们的辞职意愿。这是因为满意度提高了他们的信心、市场价值和应该得到更好待遇的感觉，使得与糟糕的经理的不匹配更加突出，让他们离职风险增加。People with high satisfaction and good relationships with managers are likely to be poached by other companies more easily as shown in Figure 1. When the relationship between employees and managers is not good, higher job satisfaction will amplify their willingness to resign. This is because satisfaction boosts their confidence, market value and the sense that they deserve better treatment, making the mismatch with bad managers more prominent and increasing their risk of leaving.

In [ ]:
# Attrition_Flag ~ Job_Satisfaction + Relationship_with_Manager + Job_Satisfaction * Relationship_with_Manager
model1_int = smf.logit(
    "Attrition_Flag ~ Job_Satisfaction * Relationship_with_Manager",
    data=df
).fit()

print(model1_int.summary())

# 预测每个样本的概率
df["pred_prob_mgr_jobsat"] = model1_int.predict(df)

line1 = (
    df.groupby(["Job_Satisfaction", "Relationship_with_Manager"])
      .agg(
          pred_prob=("pred_prob_mgr_jobsat", "mean"),
          n=("Attrition_Flag", "size")
      )
      .reset_index()
)

print(line1.head())
plt.figure(figsize=(8, 6))

for mgr_level, sub in line1.groupby("Relationship_with_Manager"):
    plt.plot(
        sub["Job_Satisfaction"],
        sub["pred_prob"],
        marker="o",
        label=f"Relationship_with_Manager={mgr_level}"
    )

plt.xlabel("Job_Satisfaction")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability\nby Job Satisfaction × Relationship_with_Manager (with interaction)")
plt.legend(title="Manager Relationship")
plt.grid(True)
plt.tight_layout()
plt.savefig("4-Predicted Attrition Probability by Job Satisfaction × Relationship_with_Manager (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()

加班对员工流失率的影响具有很强的部门特异性，因为员工会通过自身岗位设计和工作文化的视角来解读加班现象。
在财务和市场营销部门——这些工作往往受外部节奏驱动、以截止日期为导向且多为被动响应——加班通常意味着工作量失控和资源短缺，从而导致更高流失率。
但在人力资源、信息技术和销售部门，加班更多与可预测的项目周期、问题解决或绩效激励相关，这意味着这些部门的加班者可能恰恰是更投入的贡献者，他们感受到价值认同与职业安全感。
由此形成两极分化模式：在高压部门，加班是警报信号；而在项目驱动或激励导向的职能部门，加班则成为敬业度与发展机遇的标识。
The impact of overtime on attrition is highly department-specific because employees interpret overtime through the lens of their job design and work culture.
In Finance and Marketing—where work is deadline-driven, externally paced, and often reactive—overtime tends to signal unmanageable workload and resource gaps, leading to higher attrition.
But in HR, IT, and Sales, overtime is more often tied to predictable project cycles, problem-solving, or performance incentives, meaning that employees who work overtime there may actually be the more engaged contributors who feel valued and secure.
This produces the split pattern: overtime is a red flag in pressure-heavy departments, but a marker of commitment and opportunity in project- or incentive-driven functions.

In [ ]:
model2_int = smf.logit(
    "Attrition_Flag ~ C(Department) * C(Overtime)",
    data=df
).fit()

print(model2_int.summary())

# 预测每个样本的概率
df["pred_prob_dept_ot"] = model2_int.predict(df)

line2 = (
    df.groupby(["Department", "Overtime"])
      .agg(
          pred_prob=("pred_prob_dept_ot", "mean"),
          n=("Attrition_Flag", "size")
      )
      .reset_index()
)

print(line2)

plt.figure(figsize=(8, 6))

for ot, sub in line2.groupby("Overtime"):
    plt.plot(
        sub["Department"],
        sub["pred_prob"],
        marker="o",
        label=f"Overtime={ot}"
    )

plt.xlabel("Department")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability\nby Department × Overtime (with interaction)")
plt.legend(title="Overtime")
plt.grid(True)
plt.tight_layout()
plt.savefig("5-Predicted Attrition Probability by Department × Overtime (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()

加班对员工流失率的影响具有很强的部门特异性，因为员工会通过自身岗位设计和工作文化的视角来解读加班现象。 在财务和市场营销部门——这些工作往往受外部节奏驱动、以截止日期为导向且多为应急性质——加班通常意味着工作量失控和资源短缺，从而导致更高流失率。 但在人力资源、信息技术和销售部门，加班更多与可预测的项目周期、问题解决或绩效激励相关，这意味着这些部门的加班者可能恰恰是更投入的贡献者，他们感受到价值认同与职业安全感。 由此形成两极分化模式：在高压部门，加班是警报信号；而在项目驱动或激励导向的职能部门，加班则成为敬业度与发展机遇的标识。 The impact of overtime on attrition is highly department-specific because employees interpret overtime through the lens of their job design and work culture. In Finance and Marketing—where work is deadline-driven, externally paced, and often reactive—overtime tends to signal unmanageable workload and resource gaps, leading to higher attrition. But in HR, IT, and Sales, overtime is more often tied to predictable project cycles, problem-solving, or performance incentives, meaning that employees who work overtime there may actually be the more engaged contributors who feel valued and secure. This produces the split pattern: overtime is a red flag in pressure-heavy departments, but a marker of commitment and opportunity in project- or incentive-driven functions.

In [ ]:
import pandas as pd

bins = [18, 30, 40, 60] 
labels = ["18–29", "30–39", "40–59"]
df["Age_Band"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

print(df[["Age", "Age_Band"]].head())

model_age_ten = smf.logit(
    formula="Attrition_Flag ~ Years_at_Company * C(Age_Band)",
    data=df
).fit()

print(model_age_ten.summary())

df["pred_prob_age_ten"] = model_age_ten.predict(df)

line_age_ten = (
    df.groupby(["Years_at_Company", "Age_Band"])
      .agg(
          pred_prob=("pred_prob_age_ten", "mean"),
          n=("Attrition_Flag", "size")
      )
      .reset_index()
)

print(line_age_ten.head())

plt.figure(figsize=(8, 6))

for band, sub in line_age_ten.groupby("Age_Band"):
    sub_sorted = sub.sort_values("Years_at_Company")
    plt.plot(
        sub_sorted["Years_at_Company"],
        sub_sorted["pred_prob"],
        marker="o",
        label=f"Age_Band={band}"
    )

plt.xlabel("Years at Company")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability by Years_at_Company × Age_Band (with interaction)")
plt.legend(title="Age Band")
plt.grid(True)
plt.tight_layout()
plt.savefig("6-Predicted Attrition Probability by Years_at_Company × Age_Band (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()

图中显示，不同婚姻状态的员工在加班与否之间的离职概率呈现出明显不同的变化趋势：已婚与单身员工的离职概率随是否加班几乎不变，说明加班对这两个群体的影响相对有限；而离异员工则呈现出截然相反的模式，不加班时的离职风险最高，加班后反而显著下降。一个合理的解释是，离异员工在生活结构、责任分配或经济压力上可能与其他群体不同，加班带来的额外收入或更高的工作投入可能降低其离职意愿，从而形成这种“加班越多越稳定”的独特趋势。这一交互效应提醒我们，加班政策对不同员工群体的影响具有明显差异，管理者在制定人力政策时应考虑个人生活阶段与家庭背景带来的影响。
The graph shows that the turnover probabilities of employees with different marital statusing show significantly different changing trends whether they work overtime or not: The turnover probabilities of married and single employees remain almost unchanged with or without working overtime, indicating that the impact of overtime on these two groups is relatively limited. In contrast, divorced employees show a completely opposite pattern. Their risk of leaving is the highest when they do not work overtime, but it drops significantly after they work overtime. A reasonable explanation is that divorced employees may differ from other groups in terms of life structure, responsibility distribution or economic pressure. The additional income or higher work input brought by overtime may reduce their willingness to leave, thus forming this unique trend of "the more overtime, the more stable". This interaction effect reminds us that the impact of overtime policies on different employee groups varies significantly. When formulating human resource policies, managers should take into account the influence brought by personal life stages and family backgrounds.


In [ ]:
# Attrition_Flag ~ C(Marital_Status) + C(Overtime) + 交互
model4_int = smf.logit(
    "Attrition_Flag ~ C(Marital_Status) * C(Overtime)",
    data=df
).fit()

print(model4_int.summary())

# 预测每个样本的离职概率
df["pred_prob_marital_ot"] = model4_int.predict(df)

line4 = (
    df.groupby(["Marital_Status", "Overtime"])
      .agg(
          pred_prob=("pred_prob_marital_ot", "mean"),
          n=("Attrition_Flag", "size")
      )
      .reset_index()
)

print(line4)

plt.figure(figsize=(8, 6))

for ms, sub in line4.groupby("Marital_Status"):
    plt.plot(
        sub["Overtime"],
        sub["pred_prob"],
        marker="o",
        label=f"Marital_Status={ms}"
    )

plt.xlabel("Overtime")
plt.ylabel("Predicted Attrition Probability")
plt.title("Predicted Attrition Probability\nby Overtime × Marital_Status (with interaction)")
plt.legend(title="Marital_Status")
plt.grid(True)
plt.tight_layout()
plt.savefig("7-Predicted Attrition Probability by Overtime × Marital_Status (with interaction).png", dpi=300, bbox_inches="tight")
plt.show()